## Installation (GPU only)

pip install blue_sampler[gpu13] \
pip install blue_sampler[keops]

In [ ]:
!pip install --upgrade blue_sampler[gpu13]
!pip install --upgrade blue_sampler[keops]

In [ ]:
import blue_sampler as blue
from blue_sampler.run.run_nufft_keops import _nufft_pipeline_keops as sample

from pathlib import Path
import zipfile

import h5py
import numpy as np

import time

#Fancy plots
import matplotlib.pyplot as plt
colrs = [c["color"] for c in plt.rcParams["axes.prop_cycle"]]
color = lambda i: colrs[i % len(colrs)]

#output path for the files
ROOT = Path(".")

## Sampling Configuration

### Mini experiment

The **mini** configuration is intended for quick experiments.

* **Dimension:** 2 to 8
* **Number of points:** \(N = 1,024\) to \(16,384\) (powers of two)
* **runtime:** approximately 2 minutes on a GPU
* **Reference hardware:** Google Colab T4

### Full experiment

Use the **full** configuration to reproduce the full sampling domain used in the paper.

* **Dimension:** 2 to 16
* **Number of points:** \(N = 1,024\) to \(131,072\)
* **runtime:** approximately 2 hour on a GPU

To run the mini resp full experiment, use:

```text
sampling = "mini" resp "all"
```

### NUFFT vs. NUFFT+

The **Chi (χ) parameter** controls the maximum wave vector included in the gradient descent cost function (see **Torcato et al.**).

* **Chi = 0.30:** Exclusion region covers 60% of reachable wave vectors; S(k) control is very strong S(k) <= 10⁻¹⁰.
* **Chi = 0.43:** Exclusion region covers 86% of reachable wave vectors; S(k) control is weaker S(k) ≤ 10⁻⁴.


In [ ]:
sampling = "mini" #or sampling = "all"
quality = "nufft+" #or quality = "nufft"

# "nufft" is 4 time faster than "nufft+" but not so good on high frequencys

assert sampling in ("mini", "all")
assert quality in ("nufft", "nufft+")

D_MIN = 2 # dimension D
POW_MIN = 10 # number of points N = 2**pow

D_MAX = 8 if sampling == "mini" else 16
POW_MAX = 14 if sampling == "mini" else 17

print(D_MAX, POW_MAX)

#see Torcato et. al for Chi parameter, NITER is the number of gradient steps
CHI, NITER = (0.43, 150) if quality == "nufft+" else (0.3, 150)

# Crash test

In [ ]:
runtimes = {}
pow = 10
N = 2**pow
print(f"==== Test: nufft sampler (N = {N}) ====")
for D in range(D_MIN, D_MAX + 1):
    runtimes[f"D{D}"] = {}

    print(f"D={D}")

    start = time.time()

    x = sample(
        N,
        D,
        n_iter=NITER,
        verbose=0,
        Chi=CHI,
    )

    runtimes[f"D{D}"][f"N{N}"] = {f"{(time.time() - start):.2f}s"}

    if D == D_MIN:
        ax = None
    if D == 2:
        blue.plot(x)

    fig, ax = blue.plot_structure_factor(
                x, return_fig = True, label = D,
                color = color(D-D_MIN), ax = ax
              )

print(runtimes)
plt.show()

# Lets go

full pipeline


In [ ]:
runtimes = {}
for D in range(D_MIN, D_MAX + 1):
    runtimes[f"D{D}"] = {}
    print(f"\n=== Processing D={D} ===")

    npy_files = []

    # 1. Generate NPY files
    for pow in range(POW_MIN, POW_MAX + 1):
        N = 2**pow
        npy_path = ROOT / f"N{pow}_D{D}.npy"

        print(f"  Generating N={N} -> {npy_path.name}")

        start = time.time()

        x = sample(
            N,
            D,
            n_iter=NITER,
            verbose=0,
            Chi=CHI,
        )

        runtimes[f"D{D}"][f"N{N}"] = {f"{(time.time() - start):.2f}s"}

        np.save(npy_path, x)
        npy_files.append(npy_path)

    # 2. Convert NPY -> HDF5
    h5_files = []

    for npy_path in npy_files:
        h5_path = npy_path.with_suffix(".h5")

        points = np.load(npy_path)

        with h5py.File(h5_path, "w") as f:
            f.create_dataset(
                "points",
                data=points,
                dtype="float32",
            )

        h5_files.append(h5_path)

    # 3. Create D{D}.zip
    zip_path = ROOT / f"D{D}.zip"

    print(f"  Creating {zip_path.name}...")

    with zipfile.ZipFile(
        zip_path,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
    ) as z:
        for h5_path in h5_files:
            z.write(h5_path, arcname=h5_path.name)

    print(f"  Archive created: {zip_path.name}")

    # 4. Remove all intermediate NPY and HDF5 files
    for path in npy_files + h5_files:
        path.unlink()

    print("  Temporary NPY and HDF5 files removed.")
    print(f"=== D={D} completed ===")

print("\nAll dimensions completed.")

In [ ]:
print(runtimes)

## Runtimes

Sampling times on a single Google Colab T4 GPU, in seconds, for each (N, D) 

- For large N and D, runtime grows asymptotically like `2 × 10⁻¹⁰ · (N·D)²` seconds. 
- For small samples, a fixed overhead of about 0.3 s dominates.

| D | 2^10 | 2^11 | 2^12 | 2^13 | 2^14 | 2^15 | 2^16 | 2^17 | Total |
|---|---|---|---|---|---|---|---|---|---|
| 2 | 0.3 | 0.4 | 0.3 | 0.5 | 0.7 | 1.5 | 5.0 | 19 | 28 s |
| 3 | 0.2 | 0.2 | 0.3 | 0.5 | 0.9 | 2.8 | 9.2 | 37 | 51 s |
| 4 | 0.2 | 0.3 | 0.3 | 0.5 | 1.3 | 4.3 | 15 | 58 | 81 s |
| 5 | 0.2 | 0.3 | 0.4 | 0.9 | 2.0 | 6.1 | 23 | 87 | 2 min |
| 6 | 0.2 | 0.3 | 0.5 | 0.9 | 2.5 | 8.4 | 31 | 119 | 3 min |
| 7 | 0.3 | 0.3 | 0.5 | 1.1 | 3.2 | 11 | 41 | 158 | 4 min |
| 8 | 0.3 | 0.4 | 0.6 | 1.4 | 4.2 | 14 | 52 | 203 | 5 min |
| 9 | 0.3 | 0.4 | 0.7 | 1.7 | 5.1 | 17 | 64 | 249 | 6 min |
| 10 | 0.3 | 0.5 | 0.8 | 2.0 | 6.1 | 21 | 76 | 296 | 7 min |
| 11 | 0.4 | 0.6 | 1.1 | 2.9 | 8.4 | 30 | 109 | 427 | 10 min |
| 12 | 0.4 | 0.6 | 1.0 | 2.8 | 8.5 | 30 | 113 | 438 | 10 min |
| 13 | 0.4 | 0.7 | 1.4 | 3.8 | 11 | 40 | 149 | 582 | 13 min |
| 14 | 0.5 | 0.7 | 1.3 | 3.5 | 11 | 39 | 145 | 564 | 13 min |
| 15 | 0.5 | 0.9 | 1.6 | 4.7 | 15 | 53 | 195 | 766 | 17 min |
| 16 | 0.5 | 0.8 | 1.9 | 4.8 | 16 | 54 | 201 | 782 | 18 min |
| **All** | 5.1 | 7.4 | 13 | 32 | 96 | 332 | 1227 | 4785 | **1 h 48 min** |
